In [2]:
import pandas as pd, numpy as np
from datetime import date


In [3]:
surv = pd.read_excel(r"../data/private_dataF.xlsx")   #change path if needed

### Data preparation and inspection

In [3]:
surv_clean = surv.copy()

# checking the missing values --> result:  NO MISSING VALUES
missing_summary = pd.DataFrame({
    "Missing count": surv_clean.isnull().sum(),
    "Missing percentage": (surv_clean.isnull().mean() * 100).round(2)
})

print(missing_summary)

# Identify all rows with duplicate name and date of birth --> result: NO DUPLICATES
duplicates = surv_clean[
    surv_clean.duplicated(subset=["name", "dob"], keep=False)
].sort_values(["name", "dob"])

print(duplicates)

# Checking the unique values and their frequencies.
categorical_columns = [
    "sex",
    "zip",
    "evote",
    "last_voted",
    "party",
    "marital_status",
    "education",
    "citizenship"
]

for col in categorical_columns:
    if col in surv_clean.columns:
        print(f"\n--- {col} ---")
        print(surv_clean[col].value_counts(dropna=False))

                Missing count  Missing percentage
name                        0                 0.0
sex                         0                 0.0
evote                       0                 0.0
dob                         0                 0.0
zip                         0                 0.0
education                   0                 0.0
citizenship                 0                 0.0
marital_status              0                 0.0
party                       0                 0.0
Empty DataFrame
Columns: [name, sex, evote, dob, zip, education, citizenship, marital_status, party]
Index: []

--- sex ---
sex
Female    100
Male      100
Name: count, dtype: int64

--- zip ---
zip
2400    75
2200    48
2100    40
2300    37
Name: count, dtype: int64

--- evote ---
evote
0    138
1     62
Name: count, dtype: int64

--- party ---
party
Green           127
Red              67
Invalid vote      6
Name: count, dtype: int64

--- marital_status ---
marital_status
Married/separated   

### zip as str, bin age and map education

In [4]:
# make sure zip is treated as a category, not a number
surv_clean["zip"] = surv_clean["zip"].astype(str)

# making citizenship being "danish" vs. "non-danish"
surv_clean["citizenship"] = surv_clean["citizenship"].apply(lambda x: "Danish" if x == "Denmark" else "non-danish")

# changing the dob into age and binning it into 3/4 categories
date_then = date(2025, 7, 1)

surv_clean = surv_clean.copy()

surv_clean["age"] = surv_clean["dob"].apply(
    lambda dob: date_then.year - dob.year
    - ((date_then.month, date_then.day) < (dob.month, dob.day))
)

surv_clean["age_group"] = pd.cut(
    surv_clean["age"],
    bins=[18, 30, 45, 65, np.inf],
    labels=["18-29", "30-44", "45-64", "65 or older"],
    right=False,
)


# Remove rows where education is "Not stated"
surv_clean = surv_clean[surv_clean["education"] != "Not stated"].copy()

# education grouping
education_mapping = {
    "Primary education": "Low",
    "Upper secondary education": "Medium",
    "Vocational Education and Training (VET)": "Medium",
    "Short cycle higher education": "High",
    "Vocational bachelors educations": "High",
    "Bachelors programmes": "High",
    "Masters programmes": "High",
    "PhD programmes": "High"
}

surv_clean["education"] = surv_clean["education"].map(education_mapping)

print(surv_clean["education"].value_counts(dropna=False))

surv_clean

education
Medium    85
High      73
Low       38
Name: count, dtype: int64


,name,sex,evote,dob,zip,education,citizenship,marital_status,party,age,age_group
0,"Casas-Marta, Angelica",Female,0,1997-05-24,2100,Medium,Danish,Never married,Green,28,18-29
1,"el-Salahuddin, Rawda",Female,0,1964-01-03,2300,Medium,Danish,Never married,Red,61,45-64
2,"al-Haider, Saahira",Female,1,1976-01-15,2400,High,Danish,Never married,Green,49,45-64
3,"Hansen, Karan",Male,1,1960-04-14,2400,High,Danish,Never married,Red,65,65 or older
4,"Lee, Cody",Male,0,1984-10-17,2100,Medium,Danish,Never married,Green,40,30-44
...,...,...,...,...,...,...,...,...,...,...,...
195,"Rutherford, Joshua",Male,0,1940-05-15,2400,Medium,Danish,Married/separated,Green,85,65 or older
196,"Santee, Michael",Male,1,1964-08-22,2400,Medium,Danish,Married/separated,Green,60,45-64
197,"Elwood, Adrienne",Female,0,1966-04-24,2400,Medium,Danish,Married/separated,Green,59,45-64
198,"al-Jabour, Afaaf",Female,1,1993-01-08,2300,High,Danish,Married/separated,Invalid vote,32,30-44


### drop dob, age, annonymise name as direct indetifier

In [5]:
df = surv_clean.copy()
df = df.drop(columns=['dob', 'age'])
df["name"] = "*"

In [ ]:
df.to_csv("private_data_preprocessed.csv")